In [1]:
from pydantic import BaseModel
from langgraph.graph import StateGraph, START,END
from typing import Literal
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1")

class localState(BaseModel):
    name: str =  ""
    answer : str = ""
    question : str = ""

In [2]:
def coding_agent(state:localState) -> localState:
    print("Coding Agent Called")
    response = llm.invoke([{"role":"user", "content": state.question}])
    state.answer = response.content
    return state

def weather_agent(state:localState) -> localState:
    # print("Weather Agent Called")
    response = llm.invoke([{"role":""}])
    return state

def history_agent(state:localState) -> localState:
    print("History Agent Called")
    return state

In [3]:
def decider(state:localState) -> localState:    
    value = state.name.lower()

    responseQuestion = llm.invoke(f""" 
                                    You have to tell me the category of this question {value}
                                    Rule:
                                        - Category can either be "coding","weather","history"
                                        - Always give me only a single word, no extra lines or word
                                        - if question does not belong to any of these category just mention it as 'history'
                                  
                                  """)

    # if value in ["coding", "string"]:
    #     state.name = "coding"
    #     # return state
    # elif value in ["climate", "season"]:
    #     state.name = "weather"
    #     # return state
    # else:
    #     state.name = "history"
    #     # return state
    print(responseQuestion)
    state.name = responseQuestion.content
    return state
    

def router(state:localState) -> Literal["coding_agent", "weather_agent", "history_agent"]:
    return f"{state.name}_agent"
    

In [4]:
builder = StateGraph(localState)

builder.add_node("coding_agent",coding_agent)
builder.add_node("weather_agent",weather_agent)
builder.add_node("history_agent",history_agent)
builder.add_node("decider",decider)

#workflow
builder.add_edge(START,"decider")
builder.add_conditional_edges("decider",router)
builder.add_edge("coding_agent",END)
builder.add_edge("weather_agent",END)
builder.add_edge("history_agent",END)

result = builder.compile()

output = result.invoke({"name":"coding", "question":"How to swap 2 array in python"})

print(output["answer"])


content='coding' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1, 'prompt_tokens': 73, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_093eed5308', 'id': 'chatcmpl-E1yZLREvSmQRmpmsu8CAtF8756cll', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6705-ce84-7502-948d-b6891df1be7f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 73, 'output_tokens': 1, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Coding Agent Called
Swapping two arrays in Python can be done in several ways depending on whether you're using **plain lists** or **NumPy arrays**.